In [1]:
!pip install sentence-transformers scikit-learn pandas numpy

   ---------------------------------------- 0.0/739.6 kB ? eta -:--:--
   --------------------------------------- 739.6/739.6 kB 11.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/793.2 kB ? eta -:--:--
   --------------------------------------- 793.2/793.2 kB 13.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 59.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ---------------------------------------- 11.7/11.7 MB 112.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 49.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   -------------------- ------------------ 62.9/122.1 MB 316.0 MB/s eta 0:00:01
   ------------------------------- ------ 102.2/122.1 MB 253.9 MB/s eta 0:00:01
   ----------------------

In [2]:
# imports
import pandas as pd
import numpy as np
import time

from sentence_transformers import SentenceTransformer   
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [3]:
print("Imports completed")

Imports completed


In [4]:
# load the data and look at it
df = pd.read_csv("fake_job_postings.csv")   
print(df.shape)   # (rows, columns)
df.head()

(17880, 18)


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


In [5]:
# check the class balance
df["fraudulent"].value_counts()

fraudulent
0    17014
1      866
Name: count, dtype: int64

In [6]:
# merge the four text columns into one
text_cols = ["company_profile", "description", "requirements", "benefits"]
df[text_cols] = df[text_cols].fillna("")           # blanks -> empty string
df["combined_text"] = df[text_cols].agg(" ".join, axis=1)

df["combined_text"].iloc[0]   # look at the first merged post

"We're Food52, and we've created a groundbreaking and award-winning cooking site. We support, connect, and celebrate home cooks, and give them everything they need in one place.We have a top editorial, business, and engineering team. We're focused on using technology to find new and better ways to connect people around their specific food interests, and to offer them superb, highly curated information about food and cooking. We attract the most talented home cooks and contributors in the country; we also publish well-known professionals like Mario Batali, Gwyneth Paltrow, and Danny Meyer. And we have partnerships with Whole Foods Market and Random House.Food52 has been named the best food website by the James Beard Foundation and IACP, and has been featured in the New York Times, NPR, Pando Daily, TechCrunch, and on the Today Show.We're located in Chelsea, in New York City. Food52, a fast-growing, James Beard Award-winning online food community and crowd-sourced and curated recipe hub,

In [8]:
# load the Sentence Transformer model
#model = SentenceTransformer("all-MiniLM-L6-v2")
#print("Model loaded successfully") #this didn't work so i asked claude to help

# load the Sentence Transformer model
import os
from huggingface_hub import snapshot_download

local_dir = os.path.join(os.getcwd(), "all-MiniLM-L6-v2")
if not os.path.exists(local_dir):
    snapshot_download(repo_id="sentence-transformers/all-MiniLM-L6-v2", local_dir=local_dir)

model = SentenceTransformer(local_dir)
print("Model loaded successfully")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully


In [9]:
# testing on a small subset first to check the timing
subset = df["combined_text"].head(100).tolist()

start = time.time()
embeddings = model.encode(subset)
end = time.time()

print("Time taken:", end - start, "seconds")
print("Shape:", embeddings.shape)

Time taken: 6.401045560836792 seconds
Shape: (100, 384)


In [10]:
# creating embeddings for all the job posts
start = time.time()
embeddings = model.encode(df["combined_text"].tolist(), show_progress_bar=True)
end = time.time()

print("Time taken:", (end - start) / 60, "minutes")
print("Shape:", embeddings.shape)

# save the embeddings so we don't have to redo this every time
np.save("embeddings.npy", embeddings)
print("Saved embeddings.npy")

Batches:   0%|          | 0/559 [00:00<?, ?it/s]

Time taken: 5.624020640055338 minutes
Shape: (17880, 384)
Saved embeddings.npy


In [11]:
# seting up features (xs) and labels (ys)
xs = embeddings
ys = df["fraudulent"].values

# split into training and test sets 
xs_train, xs_test, ys_train, ys_test = train_test_split(xs, ys, test_size=0.20, random_state=42)

print("Training set:", xs_train.shape)
print("Test set:", xs_test.shape)

Training set: (14304, 384)
Test set: (3576, 384)


In [12]:
# training Random Forest classifier
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(xs_train, ys_train)

# checking the accuracy on train and test sets
print("Train accuracy:", accuracy_score(ys_train, rf_clf.predict(xs_train)))
print("Test accuracy:", accuracy_score(ys_test, rf_clf.predict(xs_test)))

Train accuracy: 0.9999300894854586
Test accuracy: 0.9745525727069351


In [13]:
# testing to see how well the classifier catches fake posts (not just overall accuracy)
predictions = rf_clf.predict(xs_test)
print(confusion_matrix(ys_test, predictions))

[[3395    0]
 [  91   90]]
